# 01 — Train one model (all its strategies) on Colab

**Prerequisite:** run `00_colab_setup.ipynb` once (it caches the dataset zip on Google Drive).

1. `Runtime > Change runtime type > T4 GPU`
2. `Runtime > Run all`. With `MODEL = "auto"` the next model not yet trained is chosen automatically.
   The 3 transfer-learning strategies of this model are trained one after the other.

Results are written to Google Drive (`MyDrive/road-damage-tl/runs/`) and to MLflow
(database `MyDrive/road-damage-tl/mlflow.db`, artifacts in `MyDrive/road-damage-tl/mlartifacts/`),
so nothing is lost when the Colab session ends. If the session stops in the middle, just run again:
strategies already finished are skipped. Do not run the same model on two accounts at the same time.

| MODEL | Owner | Settings |
|---|---|---|
| `scratch_cnn` | Student 2 | trained once (no strategy) |
| `alexnet` | Student 2 | default |
| `convnextv2` | Student 1 | default |
| `vit_b16` | Student 3 | default |
| `nfnet_f1` | Student 3 | heavy: batch 16 x 2 accumulation steps (set automatically) |

In [ ]:
MODEL = "auto"                 # "auto" = next model not trained yet; or scratch_cnn | alexnet | convnextv2 | vit_b16 | nfnet_f1
EPOCHS = 30
BATCH_SIZE = 32
ACCUM_STEPS = 1                # effective batch = BATCH_SIZE x ACCUM_STEPS (keep it = 32)
RETRAIN = False                # False: strategies already finished on GPU for this MODEL are skipped

REPO_URL = "https://github.com/Brahim-semlali/road-damage-tl.git"
DRIVE_ROOT = "/content/drive/MyDrive/road-damage-tl"
REPO_DIR = "/content/road-damage-tl"
RAW_DIR = "/content/data/raw"

## 1. Setup: Drive, code, dependencies, data

In [ ]:
import glob, os, subprocess
from google.colab import drive
drive.mount("/content/drive")

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log --oneline -1

!pip install -q timm albumentations mlflow

if not os.path.isdir(f"{RAW_DIR}/sih_road_dataset"):
    zips = sorted(glob.glob(f"{DRIVE_ROOT}/datasets/*.zip"))
    assert zips, ("No dataset zip in " + DRIVE_ROOT + "/datasets. If the folder road-damage-tl was SHARED with you: "
                  "drive.google.com > Shared with me > right-click road-damage-tl > Organize > Add shortcut > My Drive, "
                  "then Runtime > Restart session and Run all. Otherwise run 00_colab_setup.ipynb first.")
    os.makedirs(RAW_DIR, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", zips[0], "-d", RAW_DIR], check=True)
print(sum(len(f) for _, _, f in os.walk(RAW_DIR)), "files in", RAW_DIR)

import torch
assert torch.cuda.is_available(), (
    "NO GPU! Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all. "
    "(Training on CPU is ~20x slower and gives wrong training times.)")
print("GPU:", torch.cuda.get_device_name(0))

# Strategies already finished on GPU (found on Drive) are skipped
import json
ORDER = ["scratch_cnn", "alexnet", "convnextv2", "vit_b16", "nfnet_f1"]

def all_strategies(model):
    return ["from_scratch"] if model == "scratch_cnn" else ["feature_extraction", "partial_finetune", "full_finetune"]

def done_strategies(model):
    done = set()
    for f in glob.glob(f"{DRIVE_ROOT}/runs/{model}__*/metrics.json"):
        if json.load(open(f)).get("device", "cpu") != "cpu":
            done.add(os.path.basename(os.path.dirname(f)).split("__")[1])
    return done

for m in ORDER:
    print(f"{m:12s} done on GPU: {sorted(done_strategies(m)) or '-'}")
if MODEL == "auto":
    todo = [m for m in ORDER if set(all_strategies(m)) - done_strategies(m)]
    assert todo, "All models are trained: run 02_analysis_colab.ipynb."
    MODEL = todo[0]
STRATEGIES = all_strategies(MODEL)
if not RETRAIN:
    STRATEGIES = [s for s in STRATEGIES if s not in done_strategies(MODEL)]
if MODEL == "nfnet_f1":            # heavy model: same effective batch (32), less GPU memory
    BATCH_SIZE, ACCUM_STEPS = 16, 2
print()
print(">>> MODEL:", MODEL, "| strategies to train:", STRATEGIES, "| batch:", BATCH_SIZE, "x", ACCUM_STEPS)
if not STRATEGIES:
    print("Nothing to train for", MODEL, "-> set MODEL = 'auto' in the first cell.")

## 2. Quick sanity check (2 batches, 1 epoch, every strategy)

Catches errors (and downloads the pretrained weights) before launching the long runs.

In [ ]:
for strategy in STRATEGIES:
    !python -m src.train --model {MODEL} --strategy {strategy} --epochs 1 --max-batches 2 --batch-size {BATCH_SIZE} --data-root {RAW_DIR} --out /content/debug_runs 2>&1 | grep -E "Run:|Parameters|Epoch|Traceback|Error"

## 3. Training (one run per strategy)

In [ ]:
# MLflow SQLite database: used on the local disk (SQLite is unreliable on the Drive file
# system) and copied back to Drive after each run.
import shutil, time
DB_LOCAL, DB_DRIVE = "/content/mlflow.db", f"{DRIVE_ROOT}/mlflow.db"
if os.path.exists(DB_DRIVE):
    shutil.copy(DB_DRIVE, DB_LOCAL)

session_start = time.time()
for strategy in STRATEGIES:
    print(f"{'=' * 30} {MODEL} / {strategy} {'=' * 30}")
    !python -m src.train --model {MODEL} --strategy {strategy} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --accum-steps {ACCUM_STEPS} --data-root {RAW_DIR} --out "{DRIVE_ROOT}/runs" --mlflow-uri "sqlite:///{DB_LOCAL}" --mlflow-artifacts "{DRIVE_ROOT}/mlartifacts"
    if os.path.exists(DB_LOCAL):
        shutil.copy(DB_LOCAL, DB_DRIVE)

## 4. Results of this session

In [ ]:
import json
from IPython.display import Image, display
runs = [r for r in sorted(glob.glob(f"{DRIVE_ROOT}/runs/{MODEL}__*"), key=os.path.getmtime)
        if os.path.getmtime(r) >= session_start - 60]
for run in runs:
    print("=" * 100)
    print("Run folder:", run)
    if not os.path.exists(f"{run}/metrics.json"):
        print("-> Training did not finish: read the error in the output of cell 3.")
        continue
    m = json.load(open(f"{run}/metrics.json"))
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()})
    display(Image(f"{run}/curves.png", width=900))
    display(Image(f"{run}/confusion_matrix.png", width=450))

## 5. Summary of all runs so far

This table becomes the "Global results" table of the paper (Section IV.6).

In [ ]:
import pandas as pd
rows = []
for f in sorted(glob.glob(f"{DRIVE_ROOT}/runs/*/metrics.json")):
    m = json.load(open(f))
    if m.get("device", "cpu") == "cpu":
        continue                      # CPU runs are ignored (not comparable)
    name = os.path.basename(os.path.dirname(f)).split("__")
    rows.append({"model": name[0], "strategy": name[1], **m})
cols = ["model", "strategy", "test_accuracy", "test_precision_macro", "test_recall_macro",
        "test_f1_macro", "test_auc_macro_ovr", "params_total", "params_trainable",
        "train_time_s", "peak_gpu_mem_mb", "latency_ms", "best_epoch"]
pd.DataFrame(rows)[cols].round(4)